Added SNR estimate as 4th output channel

Also training on more difficult data: 
- d0.00-0.80
- added +/- 5 deg rotation to pipette
- more background noise


**Performance:** 

**Model:**
ImageNet1k_v2
- training resnet [-2][-2:] and [-1]
- adaptive avg pooling layer
- one dense layer with xyzw output

**Training:**
- Batch size 96
- MSE loss
- Adam, learning rate 0.001


In [ ]:
import numpy as np
from training_data import load_training_data
import matplotlib.pyplot as plt

In [ ]:
save_path = "torch_models/04_more_increased_difficulty.pth"
training_data_path = "training_data"

In [ ]:
training_data = load_training_data(training_data_path)
levels = list(training_data.levels.keys())
print(levels)

In [ ]:

# Build a pytorch model consisting of:
# - a resnet101v2 pretrained on imagenet, excluding its final classification layer
#    (the final layer has output shape (batch_size, M, N, channels)
# - a max pooling layer that takes the maximum only over channels (output shape (batch_size, M, N))
# - a dense layer with 3 one-hot outputs (output shape (batch_size, 3, onehot_size))

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

print("CUDA:", torch.cuda.is_available())

class PipetteDetector(nn.Module):
    def __init__(self):
        super(PipetteDetector, self).__init__()
        resnet101 = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        self.resnet = torch.nn.Sequential(*(list(resnet101.children())[:-2]))

        # freeze all parameters in the resnet
        for param in self.resnet.parameters():
            param.requires_grad = False
        
        # unfreeze the last block (10 layers)
        # for param in self.resnet[-1][-1].parameters():
        #     param.requires_grad = True

        # unfreeze thelast 6th-4rd blocks (30 layers) 
        for i in range(3):
            for param in self.resnet[-2][-i].parameters():
                param.requires_grad = True
        
        # unfreeze the last 3 blocks (30 layers)
        for param in self.resnet[-1].parameters():
            param.requires_grad = True
            
        # global average pooling 2d
        self.pooling = nn.AdaptiveAvgPool2d((1, 1))
        # self.pooling = nn.AdaptiveMaxPool2d((1, 1))

        self.xyzw_out = nn.LazyLinear(4)

    def forward(self, input):
        resnet_output = self.resnet(input)

        pooled = self.pooling(resnet_output)

        flat = pooled.reshape(pooled.size(0), -1)

        xyzw = self.xyzw_out(flat)
        return xyzw
    
# initialize a model just to see its structure
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from torchsummary import summary
test_model = PipetteDetector()
test_model.to(device);
# print(model)
summary(test_model, input_size=(3, 400, 400))

del test_model
torch.cuda.empty_cache()


In [ ]:
# prepare training data
from training_data import Normalizer

# decide how to transform data
class Normalizer:
    """Used to normalize and denormalize position+snr data.
    
    Input data should be a 1D numpy array with 4 elements: (z, row, col, snr)
    (z, row, col_) values are scaled and shifted to be in the range [-1, 1]
    snr is log-transformed
    """
    def __init__(self, pos_min, pos_max):
        range = np.array([pos_min+[-1], pos_max+[1]])
        diff = range[1] - range[0]
        self.scale = 2 / diff
        self.offset = range[0] + diff / 2

    def normalize(self, x):
        xn = (x - self.offset) * self.scale
        xn[:, 3] = np.log10(xn[:, 3])
        assert np.all(np.isfinite(xn))
        return xn

    def denormalize(self, x):
        x = (x / self.scale) + self.offset
        x[:, 3] = 10**(x[:, 3])
        return x

pos_min = [-100, 0, 0]
pos_max = [100, 500, 500]
pos_normalizer = Normalizer(pos_min, pos_max)

def make_image_tensor(data):
    """Convert ubyte image data to a 0.0-1.0 float32 torch tensor on the GPU."""
    normalized = (data / 255).transpose((0, 3, 1, 2)).astype(np.float32)
    return torch.tensor(normalized).to(device)
def make_position_tensor(data):
    """Convert position data to a normalized (-0.5 to 0.5) float32 torch tensor on the GPU."""
    return torch.tensor(pos_normalizer.normalize(data).astype(np.float32)).to(device)

batch_size = 96
data_generator = training_data.generator(batch_size)
# data_generator yields batches of (image, position) data
#  - images are 3D numpy arrays of shape (batch_size, height, width), with values in [0, 255]
#  - positions are 2D numpy arrays of shape (batch_size, (z, y, x)), with z values in microns and x, y in pixels 

# prepare validation data
n_validation_batches = 4
validation_data = [next(data_generator) for i in range(n_validation_batches)]
validation_images = make_image_tensor(np.concatenate([x[0] for x in validation_data]))
validation_positions = make_position_tensor(np.concatenate([x[1] for x in validation_data]))

In [ ]:
# initialize the model
model = PipetteDetector()
model.to(device)

# Define a loss function and optimizer
criterion = nn.MSELoss()  # for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)
# scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)



In [ ]:

class TrainingHistory:
    def __init__(self):
        self.history = {'i': [], 'loss': []}

    def append(self, i, prediction, target, loss):
        self.history['i'].append(i)
        self.history['loss'].append(loss)
        result = TrainingHistory.analyze_prediction(prediction, target)
        for k,v in result.items():
            if k not in self.history:
                self.history[k] = []
            self.history[k].append(v)

    def __getitem__(self, key):
        return np.array(self.history[key])
    
    def plot(self, loss_ax, accuracy_ax, label=None):
        alpha = 1.0 if 'validat' in label else 0.5
        loss_ax.plot(self.history['i'], self.history['rmse_xy'], label=label+' xy RMSE', color=(0, 0.7, 0), alpha=alpha)
        loss_ax.plot(self.history['i'], self.history['rmse_z'], label=label+' z RMSE', color=(0.7, 0, 0), alpha=alpha)
        loss_ax.plot(self.history['i'], self.history['rmse_snr'], label=label+' snr RMSE', color=(0, 0, 0.7), alpha=alpha)
        loss_ax.set_yscale('log')
        loss_ax.legend()
        accuracy_ax.plot(self.history['i'], self.history['accuracy_xy'], label=label+' xy accuracy', color=(0, 0.7, 0), alpha=alpha)
        accuracy_ax.plot(self.history['i'], self.history['accuracy_z'], label=label+' z accuracy', color=(0.7, 0, 0), alpha=alpha)
        accuracy_ax.set_ylim(0, 1)
        accuracy_ax.legend()

    @staticmethod
    def analyze_prediction(predicted_pos, target_pos, radius=0.02):
        predicted_pos = predicted_pos.cpu().detach().clone().numpy()
        target_pos = target_pos.cpu().detach().clone().numpy()
        predicted_z_sign = np.sign(predicted_pos[:, 0])
        target_z_sign = np.sign(target_pos[:, 0])
        result = {
            'rmse_xy': ((predicted_pos - target_pos)[:, 1:3]**2).mean()**0.5,
            'rmse_z': ((predicted_pos - target_pos)[:, 0]**2).mean()**0.5,
            'rmse_snr': ((predicted_pos - target_pos)[:, 3]**2).mean()**0.5,
            'n_correct_xy': (((predicted_pos - target_pos)[:, 1:3]**2).sum(axis=1)**0.5 < radius).sum(),
            'n_correct_z': (predicted_z_sign == target_z_sign).sum(),
            'batch_size': predicted_pos.shape[0],
        }
        result['accuracy_xy'] = result['n_correct_xy'] / result['batch_size']
        result['accuracy_z'] = result['n_correct_z'] / result['batch_size']
        return result


In [ ]:

# Train the model

training_history = TrainingHistory()
validation_history = TrainingHistory()
try:
    for i, data in enumerate(data_generator):
        # inputs is a 4D numpy array of shape (batch_size, height, width, channels), normalized to [0, 1]
        # labels is a 2D numpy array of shape (batch_size, 3), normalized using training_data.output_norm
        inputs, labels = data
        image_tensor = make_image_tensor(inputs)
        position_tensor = make_position_tensor(labels)

        # zero the parameter gradients (otherwise they accumulate)
        optimizer.zero_grad()

        # run model, calculate loss, and backpropagate
        output = model(image_tensor)
        loss = criterion(output, position_tensor)
        loss.backward()
        optimizer.step()
        # scheduler.step()

        # track performance over time
        training_history.append(i, output, position_tensor, loss.item())
        del image_tensor, position_tensor, output, loss
        torch.cuda.empty_cache()

        if i % 10 == 9:
            with torch.no_grad():
                val_output = model(validation_images)
                val_loss = criterion(val_output, validation_positions)
                validation_history.append(i, val_output, validation_positions, val_loss.item())

            recent_size = 10
            training_loss = training_history['loss'][-recent_size:]
            training_accuracy = training_history['accuracy_xy'][-recent_size:].mean()
            val_loss = validation_history['loss'][-1]
            validation_accuracy = validation_history['accuracy_xy'][-1]
            validation_z_accuracy = validation_history['accuracy_z'][-1]
            print(
                f'[step {i:06d}] avg sqrt loss: {(training_loss**0.5).mean():0.6f}\n'
                f'              final sqrt loss: {training_loss[-1]**0.5:0.6f}\n'
                f'              val sqrt loss: {val_loss**0.5:0.6f}\n'
                f'              val xy rmse: {validation_history["rmse_xy"][-1]:0.6f}\n'
                f'              val z rmse: {validation_history["rmse_z"][-1]:0.6f}\n'
                f'              val snr rmse: {validation_history["rmse_snr"][-1]:0.6f}\n'
                f'              training xy accuracy: {100 * training_accuracy:0.2f}%\n'
                f'              validation xy accuracy: {100 * validation_accuracy:0.2f}%\n'
                f'              validation z accuracy: {100 * validation_z_accuracy:0.2f}%\n'
            )
            del val_output, val_loss
            torch.cuda.empty_cache()
            
        if i % 100 == 99:
            print("Saving model..")
            torch.save(model.state_dict(), save_path)

except KeyboardInterrupt:
    print('Interrupted, saving..')
    torch.save(model.state_dict(), save_path)

print('Finished Training')

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
training_history.plot(ax[0], ax[1], label='training')
validation_history.plot(ax[0], ax[1], label='validation')


In [ ]:
shape = (10, len(training_data.levels))
fig, ax = plt.subplots(shape[0], shape[1], figsize=(shape[1]*3, shape[0]*3))
for i,level in enumerate(training_data.levels):
    ax[0, i].set_title(level)


for j,level in enumerate(training_data.levels):
    training_level = training_data.get_level(level)
    image, pip_pos = training_level[:shape[0]].get_arrays()
    # inds = np.arange(len(image))
    # np.random.shuffle(inds)
    # image = image[inds]
    # pip_pos = pip_pos[inds]

    # pip_pos = training_data.output_norm.denormalize(pip_pos)
    # use model in inference mode to make prediction from image
    model.eval()
    with torch.no_grad():
        image_tensor = make_image_tensor(image)
        pred = model(image_tensor).cpu().numpy()
        pred = pos_normalizer.denormalize(pred)

    for i in range(shape[0]):
        axi = ax[i, j]
        # pred = training_data.output_norm.denormalize(model.model.predict(image))
        axi.imshow(image[i], cmap='gray')
        axi.scatter(pip_pos[i, 2], pip_pos[i, 1], color='red', s=12)
        axi.scatter(pred[i, 2], pred[i, 1], color=(0.1, 1.0, 0.1), s=10)
        snr_str = f'snr={pred[i, 3]:0.3g} ({pip_pos[i, 3]:0.3g})'
        axi.axis('off')
        # add text
        axi.text(0, 0, snr_str, fontsize=8, color='blue', verticalalignment='top')

